Importing Needed Libraries - Routing to Proper Device

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

print("torch:", torch.__version__)
device = "cuda"  # Change device in the runtime settings - to GPU T4
print("device:", device)

def set_seed(seed: int = 42):
    """Make results as reproducible as possible across runs."""
    import os, random
    import numpy as np
    import torch

    os.environ["PYTHONHASHSEED"] = ":4096:8" # str(seed) - For "cpu" runtime

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Deterministic flags (safe on CPU; on GPU some ops may error if non-deterministic)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Warning: could not enable full deterministic algorithms:", e)

set_seed(42)


torch: 2.11.0+cpu
device: cuda


Importing From Kaggle API

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)

with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"YOUR_USERNAME","key":"YOUR_API_KEY"}')

os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('ucsc-cse-144-spring-2026-final-project')

print("Path to competition files:", path)

UnauthenticatedError: User is not authenticated

Dataset and DataLoader

In [2]:
# Make the images 224 × 224 for data preprocessing

# Q1: Build the data pipeline
data_dir = "./data"
batch_size = 128  # Use 128 for training
num_workers = 0  # For fully reproducible ordering across platforms

# ========== YOUR CODE STARTS HERE ==========
# TODO:
# 1) Create transforms for train and test (normalize with mean=0.1307, std=0.3081)
#    Avoid random augmentation for reproducibility
# 2) Load MNIST datasets (train and test) using datasets.MNIST()
# 3) Split training set into train (55k) and validation (5k) using random_split
# 4) Create three DataLoaders (train, val, test)
#    For train_loader, use shuffle=True

mean=0.1307
std=0.3081

train_tf = transforms.Compose([
           transforms.ToTensor(),
           transforms.Normalize(mean=(mean,), std=(std,))
])
test_tf = transforms.Compose([
          transforms.ToTensor(),
          transforms.Normalize(mean=(0.1307,), std=(0.3081,))
])

full_train = datasets.MNIST(root=data_dir,
                            train=True,
                            download=True,
                            transform=train_tf
)
test_set =   datasets.MNIST(root=data_dir,
                            train=False,
                            download=True,
                            transform=test_tf
)

train_set, val_set = random_split(full_train, [55000, 5000])

train_loader = DataLoader(dataset=train_set,
                          batch_size=64,
                          shuffle=True,
                          num_workers=0
)
val_loader =   DataLoader(dataset=val_set,
                          batch_size=64,
                          shuffle=False,
                          num_workers=0
) # Maybe change the batch size
test_loader =  DataLoader(dataset=test_set,
                          batch_size=64,
                          shuffle=False,
                          num_workers=0
) # Maybe change the batch size
# ========== YOUR CODE ENDS HERE ============

print("train/val/test:", len(train_set), len(val_set), len(test_set))

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 442kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.11MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.01MB/s]

train/val/test: 55000 5000 10000
